In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "11000-medicine-details"

# 1. Loading Required Libraries

In [2]:
library(ggthemes)
library(ggplot2)
library(reshape2)
library(plotly)
library(corrplot)
library(dplyr)
library(tidyr)
library(stringr)


Attaching package: ‘reshape2’




The following object is masked from ‘package:tidyr’:

    smiths





Attaching package: ‘plotly’




The following object is masked from ‘package:ggplot2’:

    last_plot




The following object is masked from ‘package:stats’:

    filter




The following object is masked from ‘package:graphics’:

    layout




The following object is masked from ‘package:httr’:

    config




corrplot 0.92 loaded



# 2. Load the medicine dataset

In [3]:
med <- read.csv("/kaggle/input/11000-medicine-details/Medicine_Details.csv")

# 3. Initial Data Exploration 


In [4]:
head(med)
sum(is.na(med))

,Medicine.Name,Composition,Uses,Side_effects,Image.URL,Manufacturer,Excellent.Review..,Average.Review..,Poor.Review..
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>
1,Avastin 400mg Injection,Bevacizumab (400mg),Cancer of colon and rectum Non-small cell lung cancer Kidney cancer Brain tumor Ovarian cancer Cervical cancer,Rectal bleeding Taste change Headache Nosebleeds Back pain Dry skin High blood pressure Protein in urine Inflammation of the nose,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/f5a26c491e4d48199ab116a69a969be3.jpg",Roche Products India Pvt Ltd,22,56,22
2,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),Treatment of Bacterial infections,Vomiting Nausea Diarrhea Mucocutaneous candidiasis,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/wy2y9bdipmh6rgkrj0zm.jpg",Glaxo SmithKline Pharmaceuticals Ltd,47,35,18
3,Azithral 500 Tablet,Azithromycin (500mg),Treatment of Bacterial infections,Nausea Abdominal pain Diarrhea,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/cropped/kqkouvaqejbyk47dvjfu.jpg",Alembic Pharmaceuticals Ltd,39,40,21
4,Ascoril LS Syrup,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml) + Guaifenesin (50mg/5ml),Treatment of Cough with mucus,Nausea Vomiting Diarrhea Upset stomach Stomach pain Allergic reaction Dizziness Headache Rash Hives Tremors Palpitations Muscle cramp Increased heart rate,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/3205599cc49d4073ae66cbb0dbfded86.jpg",Glenmark Pharmaceuticals Ltd,24,41,35
5,Aciloc 150 Tablet,Ranitidine (150mg),Treatment of Gastroesophageal reflux disease (Acid reflux)Treatment of Peptic ulcer disease,Headache Diarrhea Gastrointestinal disturbance,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/cropped/pn7apngctvrtweencwi1.jpg",Cadila Pharmaceuticals Ltd,34,37,29
6,Allegra 120mg Tablet,Fexofenadine (120mg),Treatment of Sneezing and runny nose due to allergiesTreatment of Allergic conditions,Headache Drowsiness Dizziness Nausea,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/fa7427131ec64163b5bbafb529df0736.jpg",Sanofi India Ltd,35,42,23


[1] 0

## Explore unique compositions in the dataset


In [5]:
unique(med$Composition)[1:50]
length(unique(med$Composition))
length(unique(med1$Manufacturer))

[1] "Bevacizumab (400mg)"                                                                                    
 [2] "Amoxycillin  (500mg) +  Clavulanic Acid (125mg)"                                                        
 [3] "Azithromycin (500mg)"                                                                                   
 [4] "Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml) + Guaifenesin (50mg/5ml)"                                
 [5] "Ranitidine (150mg)"                                                                                     
 [6] "Fexofenadine (120mg)"                                                                                   
 [7] "Pheniramine (25mg)"                                                                                     
 [8] "Donepezil (5mg)"                                                                                        
 [9] "Hydroxyzine (25mg)"                                                                                     
[10] "Phenylephrine (0.10% w/w) + Beclometasone (0.025% w/w) + Lidocaine (2.50% w/w)"                         
[11] "Montelukast (10mg) + Fexofenadine (120mg)"                                                              
[12] "Phenylephrine (5mg) + Chlorpheniramine Maleate (2mg) + Dextromethorphan Hydrobromide (10mg)"            
[13] "Phenylephrine (5mg/5ml) + Chlorpheniramine Maleate (2mg/5ml) + Dextromethorphan Hydrobromide (10mg/5ml)"
[14] "Anastrozole (1mg)"                                                                                      
[15] "Amoxycillin  (200mg) +  Clavulanic Acid (28.5mg)"                                                       
[16] "Albendazole (400mg)"                                                                                    
[17] "Clonidine (100mcg)"                                                                                     
[18] "Fexofenadine (180mg)"                                                                                   
[19] "Aceclofenac (200mg) + Rabeprazole (20mg)"                                                               
[20] "Hydroxyzine (10mg)"                                                                                     
[21] "Aceclofenac (100mg) + Paracetamol (325mg) + Serratiopeptidase (10mg)"                                   
[22] "Spironolactone (25mg)"                                                                                  
[23] "Donepezil (10mg)"                                                                                       
[24] "Donepezil (5mg) + Memantine (5mg)"                                                                      
[25] "Camphor (0.01% w/v) + Menthol (0.005% w/v) + Naphazoline (0.05% w/v) + Phenylephrine (0.12% w/v)"       
[26] "Camylofin (25mg) + Paracetamol (300mg)"                                                                 
[27] "Ambroxol (15mg/5ml) + Salbutamol (1mg/5ml)"                                                             
[28] "Phenylephrine (5mg/5ml) + Chlorpheniramine Maleate (2mg/5ml) + Dextromethorphan Hydrobromide (15mg/5ml)"
[29] "Aceclofenac (100mg) + Paracetamol (325mg)"                                                              
[30] "Erythromycin (500mg)"                                                                                   
[31] "Salbutamol (2mg/5ml)"                                                                                   
[32] "Ticagrelor (90mg)"                                                                                      
[33] "Vitamin D3 (600000IU)"                                                                                  
[34] "Alfuzosin (10mg)"                                                                                       
[35] "Azithromycin (200mg/5ml)"                                                                               
[36] "Fexofenadine (30mg/5ml)"                                                                                
[37] 

[1] 3358

ERROR: Error in eval(expr, envir, enclos): object 'med1' not found


Note about data structure: Most medicines contain multiple active ingredients separated by "+" signs. This creates a challenge for analysis because we cannot directly associate specific uses and side effects with individual ingredients when they appear in combinations.

## Dataset summary and structure


In [ ]:
summary(med)
str(med)

## Remove unnecessary column (column 5)


In [ ]:
med1 <- med[,-5]

## Preview cleaned dataset


In [ ]:
head(med1)


# 4. Review Analysis: Correlation Between Review Types
Calculate correlation between different review types


In [ ]:
cor_matrix <- cor(med1[, c("Excellent.Review..", "Average.Review..", "Poor.Review..")])


Visualize correlation matrix


In [ ]:
corrplot(cor_matrix, 
         method = "color",
         type = "upper",
         order = "hclust",
         addCoef.col = "black",
         tl.col = "black",
         tl.srt = 45,
         number.cex = 0.8,
         diag = FALSE)

Interpretation: The analysis reveals a weak negative correlation between average reviews and both poor and excellent reviews. However, there is a strong negative correlation between poor and excellent reviews, which aligns with expectations - medicines that receive excellent reviews tend to receive fewer poor reviews.


# 5. Manufacturer Performance Analysis
## Top Manufacturers by Excellent Reviews

Aggregate excellent reviews by manufacturer

In [ ]:
manufacturer_excellent <- med1 %>%      
  group_by(Manufacturer) %>%
  summarise(Total_Excellent = sum(Excellent.Review..))

Identify top 50 excellent manufacturers


In [ ]:
top_50_excellent <- manufacturer_excellent %>%
  arrange(desc(Total_Excellent)) %>%
  slice(1:50)

Create visualization

In [ ]:
plot_excellent <- ggplot(top_50_excellent, 
                         aes(x = reorder(Manufacturer, -Total_Excellent), 
                             y = Total_Excellent)) +
  geom_col(fill = "blue", colour = "black") +
  labs(title = "Total Excellent Reviews per Manufacturer", 
       x = "Manufacturer", 
       y = "Total Excellent Reviews") +
  theme_solarized() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(plot_excellent)

## Top Manufacturers by Average Reviews

Aggregate average reviews by manufacturer

In [ ]:
manufacturer_average <- med1 %>%
  group_by(Manufacturer) %>%
  summarise(Total_Average = sum(Average.Review..))

Identify top 50 average manufacturers

In [ ]:
top_50_average <- manufacturer_average %>%
  arrange(desc(Total_Average)) %>%
  slice(1:50)

Create visualization

In [ ]:
plot_average <- ggplot(top_50_average, 
                      aes(x = reorder(Manufacturer, -Total_Average), 
                          y = Total_Average)) +
  theme_minimal() +
  geom_col(fill = "skyblue", colour = "black") +
  labs(x = "Manufacturer", 
       y = "Total Average Reviews", 
       title = "Total Average Reviews per Manufacturer") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(plot_average)

## Top Manufacturers by Poor Reviews
Aggregate poor reviews by manufacturer

In [ ]:
manufacturer_poor <- med1 %>%
  group_by(Manufacturer) %>%
  summarize(Total_Poor = sum(Poor.Review..))

Identify top 50 average manufacturers


In [ ]:
top_50_poor <- manufacturer_poor %>%
  arrange(desc(Total_Poor)) %>%
  slice(1:50)

Create visualization


In [ ]:
plot_poor <- ggplot(top_50_poor, 
                   aes(x = reorder(Manufacturer, -Total_Poor), 
                       y = Total_Poor)) +
  theme_minimal() +
  labs(x = "Manufacturer", 
       y = "Total Poor Reviews", 
       title = "Total Poor Reviews per Manufacturer") +
  geom_col(fill = "Lightcoral", colour = "black") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(plot_poor)

Observation: Sun Pharma, Intas, and Cipla consistently appear at the top across all review categories. This pattern suggests that review counts may be influenced by manufacturing volume rather than quality alone.


# 6. Calculate number of drugs per manufacturer


In [ ]:
manufacturer_frequency <- med1 %>%
  count(Manufacturer) %>%
  arrange(desc(n))

top_50_manufacturers <- manufacturer_frequency %>%
  arrange(desc(n)) %>%
  slice(1:50)

plot_volume <- ggplot(top_50_manufacturers, 
                     aes(x = reorder(Manufacturer, -n), y = n)) +
  theme_minimal() +
  geom_col(fill = "red", colour = "Thistle") +
  labs(x = "Manufacturer", 
       y = "Number of Drugs", 
       title = "Drug Count per Manufacturer") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(plot_volume)

# 7. Normalized Performance Metrics

## Calculate normalized poor review score (Poor Reviews per Drug)

In [ ]:
poor_review_score <- med1 %>%
  group_by(Manufacturer) %>%
  summarise(
    Total_Medicines = n(),
    Total_Poor_Reviews = sum(`Poor.Review..`, na.rm = TRUE)
  ) %>%
  mutate(
    Poor_Review_Score = Total_Poor_Reviews / Total_Medicines
  ) %>%
  arrange(desc(Total_Medicines)) %>%
  slice(1:50) %>%
  arrange(Poor_Review_Score)

poor_review_score

Visualize poor review scores


In [ ]:
poor_score_plot <- ggplot(poor_review_score, 
                         aes(x = Total_Medicines, 
                             y = Poor_Review_Score, 
                             size = Total_Poor_Reviews, 
                             text = paste("Manufacturer:", Manufacturer))) +
  theme_bw() +
  geom_point(alpha = 0.7, colour = "red") +
  labs(x = "Number of Medicines", 
       y = "Poor Review Score", 
       title = "Manufacturer Performance: Poor Reviews per Drug",
       size = "Total Poor Reviews")

ggplotly(poor_score_plot, tooltip = "text")

Key Insight: Novartis, USV, and Glaxo have the lowest poor review scores per medicine. Sun Pharma, Intas, and Cipla have moderate scores despite high total poor reviews, because they manufacture many drugs.


## Calculate normalized excellent review score (Excellent Reviews per Drug)


In [ ]:
excellent_review_score <- med1 %>%
  group_by(Manufacturer) %>%
  summarise(
    Total_Medicines = n(),
    Total_Excellent_Reviews = sum(Excellent.Review.., na.rm = TRUE)
  ) %>%
  mutate(
    Excellent_Review_Score = Total_Excellent_Reviews / Total_Medicines
  ) %>%
  arrange(desc(Total_Medicines)) %>%
  slice(1:50)

excellent_review_score

Visualize excellent review scores


In [ ]:
excellent_score_plot <- ggplot(excellent_review_score, 
                              aes(x = Total_Medicines,
                                  y = Excellent_Review_Score,
                                  size = Total_Excellent_Reviews,
                                  text = paste("Manufacturer:", Manufacturer))) +
  geom_point(alpha = 0.5, colour = "green") +
  labs(x = "Number of Medicines", 
       y = "Excellent Review Score", 
       title = "Manufacturer Performance: Excellent Reviews per Drug",
       size = "Total Excellent Reviews") +
  theme_classic()

ggplotly(excellent_score_plot, tooltip = "text")

## Calculate normalized average review score (Average Reviews per Drug)


In [ ]:
average_review_score <- med1 %>%
  group_by(Manufacturer) %>%
  summarise(
    Total_Medicines = n(),
    Total_Average_Reviews = sum(Average.Review.., na.rm = TRUE)
  ) %>%
  mutate(
    Average_Review_Score = Total_Average_Reviews / Total_Medicines
  ) %>%
  arrange(desc(Total_Medicines)) %>%
  slice(1:50)

average_review_score

Visualize average review scores


In [ ]:
average_score_plot <- ggplot(average_review_score, 
                            aes(x = Total_Medicines,
                                y = Average_Review_Score,
                                size = Total_Average_Reviews,
                                text = paste("Manufacturer:", Manufacturer))) +
  geom_point(alpha = 0.7, colour = "blue") +
  labs(x = "Number of Medicines", 
       y = "Average Review Score", 
       title = "Manufacturer Performance: Average Reviews per Drug",
       size = "Total Average Reviews") +
  theme_classic()

ggplotly(average_score_plot, tooltip = "text")

# 8. Composition Analysis
Top 50 Medicine Compositions (Combined active Ingredients)
## Analyze combined compositions


In [ ]:
top_compositions <- med1 %>%
  group_by(Composition) %>%
  summarise(Frequency = n()) %>%
  arrange(desc(Frequency)) %>%
  slice(1:50)

composition_plot <- ggplot(top_compositions, 
                          aes(x = reorder(Composition, -Frequency), 
                              y = Frequency)) +
  geom_col(fill = "green") +
  labs(x = "Composition", 
       y = "Frequency", 
       title = "Top 50 Medicine Compositions") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(composition_plot)

Observation: Luliconazole (an antifungal) appears as the most common composition. This is unexpected since fungal infections are less common than lifestyle diseases. This pattern may result from how combination drugs are reported.


## Top 50 Individual Active Ingredients (with doses)

Separate combined compositions into individual ingredients


In [ ]:
individual_ingredients <- med1 %>%
  separate_rows(Composition, sep = "\\+") %>%
  mutate(Composition = str_trim(Composition)) %>%
  group_by(Composition) %>%
  summarise(Frequency = n()) %>%
  arrange(desc(Frequency)) %>%
  slice(1:50)

ingredients_plot <- ggplot(individual_ingredients, 
                          aes(x = reorder(Composition, -Frequency), 
                              y = Frequency)) +
  theme_minimal() +
  geom_col(fill = "red", colour = "purple") +
  labs(x = "Active Ingredient", 
       y = "Frequency", 
       title = "Top 50 Active Ingredients (with doses)") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(ingredients_plot)

## Top 50 Active Ingredients (doses removed)

Clean ingredient names by removing doses


In [ ]:
clean_ingredients <- med1 %>%
  separate_rows(Composition, sep = "\\+") %>%
  mutate(Composition = str_trim(Composition)) %>%
  mutate(Salt = str_remove_all(Composition, "\\s*\\([^)]*\\)")) %>%
  group_by(Salt) %>%
  summarise(Frequency = n()) %>%
  arrange(desc(Frequency)) %>%
  slice_head(n = 50)

clean_ingredients_plot <- ggplot(clean_ingredients, 
                                aes(x = reorder(Salt, -Frequency), 
                                    y = Frequency)) +
  theme_minimal() +
  geom_col(fill = "red", colour = "purple") +
  labs(x = "Active Ingredient", 
       y = "Frequency", 
       title = "Top 50 Active Ingredients (doses removed)") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggplotly(clean_ingredients_plot)

Key Finding: After cleaning the data, lifestyle disease medications (example: Metformin for diabetes, Telmisartan for hypertension) dominate the top positions, which aligns with current healthcare trends.


# 9. Medicine Usage Analysis

Analyze medicine uses

In [ ]:
top_uses <- med1 %>%
  group_by(Uses) %>%
  summarise(Frequency = n()) %>%
  arrange(desc(Frequency)) %>%
  slice(1:50)

uses_plot <- ggplot(top_uses, 
                   aes(x = reorder(Uses, Frequency), 
                       y = Frequency,
                       text = Uses)) +
  geom_col(fill = "green") +
  coord_flip() +
  labs(x = "Uses", 
       y = "Frequency", 
       title = "Top 50 Medicine Uses") +
  theme_classic() +
  theme(axis.text.y = element_blank(),
        axis.ticks.y = element_blank())

ggplotly(uses_plot, tooltip = "text")

Conclusion: The top medicine uses predominantly address lifestyle diseases (diabetes, hypertension) and infections, reflecting major contemporary health concerns.